In [3]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import warnings
warnings.filterwarnings("ignore")

builder = (
    SparkSession.builder
    .appName("delta-minio-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.2.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ])
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dcf8340f-737c-4345-9de5-dd4602302e0f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.1.0/delta-spark_2.12-3.1.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.1.0!delta-spark_2.12.jar (418ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.1.0/delta-storage-3.1.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.1.0!delta-storage.jar (48ms)
:: resolution report :: resolve 385ms :: artifacts dl 469ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from 

In [1]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

REPO_ROOT = Path("/workspace/rltm_bi_pltfrm/")
JOB_PATH = REPO_ROOT / "jobs" / "ingestion" / "ingest_to_bronze.py"

assert JOB_PATH.exists(), f"Canonical job not found: {JOB_PATH}"

PACKAGES = ",".join([
    "io.delta:delta-spark_2.12:3.2.0",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
])

cmd = (
    f'spark-submit '
    f'--packages {PACKAGES} '
    f'--conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" '
    f'--conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" '
    f'--conf "spark.ui.showConsoleProgress=false" '
    f'{JOB_PATH} --run-once'
)

print("Executing:", cmd)
os.system(cmd)

Executing: spark-submit --packages io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 --conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" --conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" --conf "spark.ui.showConsoleProgress=false" /workspace/rltm_bi_pltfrm/jobs/ingestion/ingest_to_bronze.py --run-once
:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0997140a-9af9-4a80-ae29-5fdaec4bed39;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (443ms)
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFU

Wrote 1 row to s3a://lakehouse/bronze/gold_price_events | symbol=XAU | price_usd=4492.200195


0

In [5]:
bronze_path = "s3a://lakehouse/bronze/gold_price_events"

df = spark.read.format("delta").load(bronze_path)

print("row_count =", df.count())
df.printSchema()

26/03/22 02:58:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/22 02:59:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 4:==========>                                              (9 + 26) / 50]

row_count = 14
root
 |-- symbol: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_event_ts: timestamp (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- bid_usd: double (nullable = true)
 |-- ask_usd: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- payload_json: string (nullable = true)
 |-- api_status: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ingest_year: integer (nullable = true)
 |-- ingest_month: integer (nullable = true)
 |-- ingest_day: integer (nullable = true)
 |-- ingest_hour: integer (nullable = true)



In [6]:
df.select(
    "event_id",
    "symbol",
    "source_name",
    "source_event_ts",
    "ingestion_ts",
    "price_usd",
    "currency",
    "api_status"
).orderBy("ingestion_ts", ascending=False).show(20, truncate=False)

+----------------------------------------------------------------+------+------------+-------------------+--------------------------+-----------+--------+----------+
|event_id                                                        |symbol|source_name |source_event_ts    |ingestion_ts              |price_usd  |currency|api_status|
+----------------------------------------------------------------+------+------------+-------------------+--------------------------+-----------+--------+----------+
|a88a15dc1aa377748fc9d7859d2b9bf283ec0cc5ea0b4588dce794fbefb88c26|XAU   |gold_api_com|2026-03-22 02:56:10|2026-03-22 02:56:11.636362|4492.200195|USD     |OK        |
|4b491db8309c4ddddac4e42a44bbfd39313c8d4c1ef4ad3568a17f535da12971|XAU   |gold_api_com|2026-03-21 17:31:10|2026-03-21 17:31:13.580731|4492.200195|USD     |OK        |
|e7ac851c1bd23e8de0a2e8d0c33f47144464957fbcea23bf4a284fdea0fedf1c|XAU   |gold_api_com|2026-03-21 17:18:10|2026-03-21 17:18:55.8226  |4492.200195|USD     |OK        |
|e0f